# Kaggle Direct Local LLM Runner

This notebook runs the ARC agent with a local Hugging Face model loaded directly through `transformers`.
Use this when `vllm` is not installed or you want a simpler fully-offline Kaggle path.

In [ ]:
import shutil
from pathlib import Path

def looks_like_repo(path: Path) -> bool:
    return path.is_dir() and (path / 'main.py').exists() and (path / 'agents').exists()

def detect_repo_source() -> Path:
    candidates = [
        Path.cwd(),
        Path('/kaggle/working/ARC-AGI-3-Agents'),
        Path('/kaggle/input/datasets/suminshim/llm-v1/ARC-AGI-3-Agents-main'),
    ]
    for base in candidates:
        if looks_like_repo(base):
            return base
    for root in [Path('/kaggle/working'), Path('/kaggle/input')]:
        if not root.exists():
            continue
        for child in root.rglob('*'):
            if looks_like_repo(child):
                return child
    raise FileNotFoundError('Could not auto-detect repo dir under /kaggle/working or /kaggle/input')

def prepare_writable_repo(src: Path) -> Path:
    if str(src).startswith('/kaggle/working'):
        return src
    dst = Path('/kaggle/working') / src.name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    return dst

REPO_SOURCE = detect_repo_source()
REPO_DIR = prepare_writable_repo(REPO_SOURCE)
MODEL_PATH = Path('/kaggle/input/your-model-dir')
WHEELS_DIR = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels')
GAME_ID = 'ls20'
AGENT_NAME = 'directlocalllm'
MAX_FRAME_CHARS = 2500
DIRECT_DTYPE = 'auto'
DIRECT_DEVICE_MAP = 'auto'
DIRECT_LOAD_IN_4BIT = False
DIRECT_TRUST_REMOTE_CODE = False
DIRECT_MAX_NEW_TOKENS = 160
DIRECT_TEMPERATURE = 0.0

assert REPO_DIR.exists(), f'Repo not found: {REPO_DIR}'
assert MODEL_PATH.exists(), f'Model path not found: {MODEL_PATH}'
assert WHEELS_DIR.exists(), f'Wheels dir not found: {WHEELS_DIR}'
print('repo source:', REPO_SOURCE)
print('repo:', REPO_DIR)
print('model:', MODEL_PATH)
print('wheels:', WHEELS_DIR)

In [ ]:
import os
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())

## Install ARC wheels from the competition dataset

This is the important offline step for `arc_agi`. It installs wheels provided by the competition dataset into Kaggle's Python environment.

In [ ]:
import subprocess
import sys

install_cmd = [
    sys.executable,
    '-m',
    'pip',
    'install',
    '--no-index',
    '--find-links',
    str(WHEELS_DIR),
    'arc-agi',
    'python-dotenv',
    'wrapt',
]
print('running:', ' '.join(install_cmd))
subprocess.run(install_cmd, check=True)
print('offline wheel install complete')

## Optional dependency checks

In [ ]:
import shutil
import subprocess

for tool in ['python', 'uv']:
    print(tool, '->', shutil.which(tool))

for mod in ['torch', 'transformers', 'accelerate']:
    try:
        __import__(mod)
        print(mod, 'OK')
    except Exception as exc:
        print(mod, 'ERR', exc)

## Make `uv` see Kaggle's preinstalled packages

Kaggle often has `torch` and `transformers` in the notebook image, but the repo's `.venv` cannot see them by default.
This cell updates `.venv/pyvenv.cfg` so `uv run` can import those packages offline without recreating the environment.

In [ ]:
import subprocess
from pathlib import Path

venv_dir = REPO_DIR / '.venv'
cfg_path = venv_dir / 'pyvenv.cfg'
if not cfg_path.exists():
    subprocess.run(['uv', 'venv', '--python', '3.12', str(venv_dir)], check=True)
    print('created .venv at', venv_dir)
text = cfg_path.read_text()
if 'include-system-site-packages = true' not in text:
    if 'include-system-site-packages = false' in text:
        text = text.replace('include-system-site-packages = false', 'include-system-site-packages = true')
    else:
        text += '\ninclude-system-site-packages = true\n'
    cfg_path.write_text(text)
print(cfg_path.read_text())
print('updated .venv to include Kaggle system site-packages')

## Write `.env` for offline direct execution

In [ ]:
from textwrap import dedent

env_text = dedent(f'''
DEBUG=False
MPLBACKEND=Agg
RECORDINGS_DIR=recordings
SCHEME=http
HOST=127.0.0.1
PORT=8000
ARC_BASE_URL=https://three.arcprize.org/
OPERATION_MODE=local
ENVIRONMENTS_DIR=environment_files
LLM_MAX_FRAME_CHARS={MAX_FRAME_CHARS}
AGENTOPS_API_KEY=
ARC_API_KEY=
DIRECT_LLM_MODEL_PATH={MODEL_PATH}
DIRECT_LLM_DEVICE_MAP={DIRECT_DEVICE_MAP}
DIRECT_LLM_DTYPE={DIRECT_DTYPE}
DIRECT_LLM_LOAD_IN_4BIT={str(DIRECT_LOAD_IN_4BIT)}
DIRECT_LLM_TRUST_REMOTE_CODE={str(DIRECT_TRUST_REMOTE_CODE)}
DIRECT_LLM_MAX_NEW_TOKENS={DIRECT_MAX_NEW_TOKENS}
DIRECT_LLM_TEMPERATURE={DIRECT_TEMPERATURE}
''').strip() + '\n'

Path('.env').write_text(env_text)
print(Path('.env').read_text())

## Optional model folder sanity check

In [ ]:
from pathlib import Path

for path in sorted(MODEL_PATH.glob('*'))[:20]:
    print(path.name)

## Run the direct local agent

In [ ]:
import os
import subprocess
from pathlib import Path

python_bin = REPO_DIR / '.venv' / 'bin' / 'python'
if not python_bin.exists():
    python_bin = Path('/usr/bin/python3')

run_cmd = [str(python_bin), 'main.py', '--agent', AGENT_NAME, '--game', GAME_ID]
print('running:', ' '.join(run_cmd))
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR)
result = subprocess.run(run_cmd, text=True, capture_output=True, cwd=REPO_DIR, env=env)
print('returncode:', result.returncode)
print(result.stdout)
if result.stderr:
    print('--- stderr ---')
    print(result.stderr)

## Inspect logs

In [ ]:
from pathlib import Path

path = Path('logs.log')
print(f'===== {path} =====')
if path.exists():
    text = path.read_text(errors='ignore')
    print(text[-12000:])
else:
    print('missing')